In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
from pyspark.sql.functions import desc

In [0]:
#movie_df = spark.read.parquet(f"{silver_folder_path}/movies").filter("release_date >= '2000-01-01'")
#movie_df = spark.read.table("movie_silver.movies").filter("release_date >= '2000-01-01'")
movie_df = spark.read.table("movie_silver.movies").filter(f"file_date >= '{v_file_date}'")

In [0]:
#movie_language_df = spark.read.parquet(f"{silver_folder_path}/movie_language")
movie_language_df = spark.read.table("movie_silver.movies_languages").filter(f"file_date >= '{v_file_date}'")

In [0]:
#language_df = spark.read.parquet(f"{silver_folder_path}/language")
language_df = spark.read.table("movie_silver.languages")

In [0]:
#movie_genre_df = spark.read.parquet(f"{silver_folder_path}/movies_genre")
movie_genre_df = spark.read.table("movie_silver.movies_genre").filter(f"file_date >= '{v_file_date}'")

In [0]:
#genre_df = spark.read.parquet(f"{silver_folder_path}/genre")
genre_df = spark.read.table("movie_silver.genre")

In [0]:
movie_final_df = movie_df \
            .join(movie_language_df, movie_df.movie_id == movie_language_df.movie_id, "inner") \
            .join(language_df, movie_language_df.language_id == language_df.language_id, "inner") \
            .join(movie_genre_df, movie_df.movie_id == movie_genre_df.movie_id, "inner") \
            .join(genre_df, movie_genre_df.genre_id == genre_df.genre_id, "inner") \
            .select (movie_df.movie_id, movie_df.title, movie_df.duration_time, movie_df.release_date, movie_df.vote_average, language_df.language_name, language_df.language_id, genre_df.genre_name, genre_df.genre_id).orderBy(desc("release_date"))

In [0]:
movie_filter_df = movie_final_df.filter("year_release_date >= 2000")

In [0]:
display(movie_filter_df)

In [0]:
from pyspark.sql.functions import lit

In [0]:
result_df = movie_filter_df.withColumn("created_date", lit(v_file_date))

In [0]:
display(result_df)

####Escribir los datos en el DataLake en formato "Parquet"

In [0]:
#overwrite_partition("movie_gold", "results_movies_genre_language", "created_date", v_file_date)

In [0]:
#result_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_movies_genre_language")
#result_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_movies_genre_language")
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.genre_id = src.genre_id AND tgt.created_date = src.created_date'
merge_delta_lake (result_df, "movie_gold", "results_movies_genre_language", merge_condition, "created_date")

In [0]:
%sql
SELECT *
FROM movie_gold.results_movies_genre_language

In [0]:
#display(spark.read.parquet(f"{gold_folder_path}/results_movies_genre_language"))